# Italic Detection Visualiser

Visualize the output of `italic_detection.py`: per-character italic labels.

**Views available:**
1. Document summary — threshold, percentages, histogram.
2. Word-level view — words colored by italic/round classification.
3. Character-level view — embedded char images grouped by italic label.
4. Browse pages.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────

IMAGE_ROOT  = r"../data/corpus-1/imgs"
JSON_ROOT   = r"../data/corpus-1/charnet"

DOCUMENT    = "BNE_1001_615_T-55281-18"
PAGE        = "page_10"

In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
import numpy as np

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 8,
    "axes.titlesize": 11,
})

In [ ]:
# ── Load document-level italic labels ────────────────────────────────

doc_dir = Path(JSON_ROOT) / DOCUMENT
italic_path = doc_dir / "italic_labels.npz"

assert italic_path.exists(), f"Italic labels not found: {italic_path}"

italic_data = np.load(str(italic_path), allow_pickle=True)

char_italic_all   = italic_data["char_italic"]        # (N_total,) int8
page_char_counts  = italic_data["page_char_counts"]   # (P,) int32
page_names        = italic_data["page_names"]         # (P,) str
threshold         = float(italic_data["threshold"][0])
n_valid_strokes   = int(italic_data["n_valid_strokes"][0])

print(f"Document: {DOCUMENT}")
print(f"Pages: {len(page_names)}")
print(f"Total characters: {len(char_italic_all)}")
print(f"Threshold: {threshold:.2f}°")
print(f"Valid strokes used: {n_valid_strokes}")

---
## 1. Document summary

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────

n_italic = char_italic_all.sum()
n_round = len(char_italic_all) - n_italic
pct_italic = 100 * n_italic / len(char_italic_all) if len(char_italic_all) > 0 else 0

print(f"Italic characters: {n_italic:,} ({pct_italic:.1f}%)")
print(f"Round characters:  {n_round:,} ({100 - pct_italic:.1f}%)")

# Per-page breakdown
print(f"\nPer-page breakdown:")
offset = 0
for pname, pcount in zip(page_names, page_char_counts):
    if pcount == 0:
        print(f"  {pname:20s}  0 chars")
        continue
    page_italic = char_italic_all[offset:offset + pcount]
    n_it = page_italic.sum()
    pct = 100 * n_it / pcount
    print(f"  {pname:20s}  {pcount:4d} chars, {n_it:4d} italic ({pct:5.1f}%)")
    offset += pcount

In [ ]:
# ── Stroke histogram with threshold ───────────────────────────────────

# Collect all strokes from all pages
all_strokes = []
for pname in page_names:
    npz_path = doc_dir / f"{pname}_orientation.npz"
    if npz_path.exists():
        data = np.load(str(npz_path), allow_pickle=True)
        strokes = data["word_stroke_orientations"]
        all_strokes.append(strokes)

all_strokes = np.concatenate(all_strokes) if all_strokes else np.array([])
valid_strokes = all_strokes[(all_strokes != -360) & ~np.isnan(all_strokes)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(valid_strokes, bins=50, color="steelblue", edgecolor="white", alpha=0.7)
ax.axvline(threshold, color="red", linestyle="--", linewidth=2, label=f"Threshold: {threshold:.2f}°")
ax.set_xlabel("Stroke orientation (degrees)")
ax.set_ylabel("Word count")
ax.set_title(f"Stroke distribution — {DOCUMENT}")
ax.legend()
fig.tight_layout()
plt.show()

---
## 2. Word-level view

Words colored by classification: **red** = italic, **blue** = round.

In [ ]:
# ── Load page data ────────────────────────────────────────────────────

img_path  = Path(IMAGE_ROOT) / DOCUMENT / f"{PAGE}.png"
json_path = doc_dir / f"{PAGE}.json"
npz_path  = doc_dir / f"{PAGE}_orientation.npz"

for p, label in [(img_path, "Image"), (json_path, "JSON"), (npz_path, ".npz")]:
    assert p.exists(), f"{label} not found: {p}"

img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)

with open(json_path) as f:
    words = json.load(f)

page_data = np.load(str(npz_path), allow_pickle=True)
word_stroke_orientations = page_data["word_stroke_orientations"]
char_word_idx = page_data["char_word_idx"]
char_imgs = page_data["char_imgs"]
char_labels_ocr = page_data["char_labels"]

n_words = len(words)
print(f"Page: {PAGE}")
print(f"Words: {n_words}, Characters: {len(char_imgs)}")

In [ ]:
# ── Get italic labels for this page ───────────────────────────────────

# Find offset for this page in the document-level array
page_idx = list(page_names).index(PAGE)
offset = int(page_char_counts[:page_idx].sum())
n_chars_page = int(page_char_counts[page_idx])

char_italic_page = char_italic_all[offset:offset + n_chars_page]

# Per-word italic classification (majority vote of its chars)
word_is_italic = np.zeros(n_words, dtype=bool)
for wi in range(n_words):
    mask = char_word_idx == wi
    if mask.sum() > 0:
        word_is_italic[wi] = char_italic_page[mask].mean() >= 0.5

n_italic_words = word_is_italic.sum()
print(f"Italic words: {n_italic_words}/{n_words} ({100*n_italic_words/n_words:.1f}%)")

In [ ]:
# ── Full-page word overlay ────────────────────────────────────────────

COLOR_ITALIC = "#e74c3c"  # red
COLOR_ROUND  = "#3498db"  # blue

fig, ax = plt.subplots(figsize=(20, 20))
ax.imshow(img_rgb)

for wi, word in enumerate(words):
    t, b, l, r = word["tblr"]
    color = COLOR_ITALIC if word_is_italic[wi] else COLOR_ROUND
    rect = Rectangle((l, t), r - l, b - t,
                      linewidth=1.5, edgecolor=color, facecolor="none")
    ax.add_patch(rect)

ax.set_axis_off()
ax.set_title(f"{DOCUMENT} / {PAGE} — Words: red=italic, blue=round", fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# ── Zoom controls ─────────────────────────────────────────────────────
X_MIN, X_MAX = 0, 1500
Y_MIN, Y_MAX = 700, 1200

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
ax.imshow(img_rgb)

for wi, word in enumerate(words):
    t, b, l, r = word["tblr"]
    color = COLOR_ITALIC if word_is_italic[wi] else COLOR_ROUND
    rect = Rectangle((l, t), r - l, b - t,
                      linewidth=2, edgecolor=color, facecolor="none")
    ax.add_patch(rect)

ax.set_xlim(X_MIN, X_MAX)
ax.set_ylim(Y_MAX, Y_MIN)
ax.set_axis_off()
ax.set_title(f"Zoom — red=italic, blue=round", fontweight="bold")
fig.tight_layout()
plt.show()

---
## 3. Character-level view

Embedded character images grouped by italic classification.

In [ ]:
# ── Separate italic vs round character images ─────────────────────────

italic_mask = char_italic_page == 1
round_mask = char_italic_page == 0

italic_imgs = char_imgs[italic_mask]
italic_labels = char_labels_ocr[italic_mask]
round_imgs = char_imgs[round_mask]
round_labels = char_labels_ocr[round_mask]

print(f"Italic chars: {len(italic_imgs)}")
print(f"Round chars:  {len(round_imgs)}")

In [ ]:
# ── Grid of italic characters ─────────────────────────────────────────
MAX_SHOW = 100
n_show = min(len(italic_imgs), MAX_SHOW)
n_cols = 20
n_rows = max(1, int(np.ceil(n_show / n_cols)))

if n_show > 0:
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 0.8, n_rows * 1.0))
    axes = np.atleast_2d(axes)
    for idx in range(n_rows * n_cols):
        r, c = divmod(idx, n_cols)
        ax = axes[r, c]
        if idx < n_show:
            ax.imshow(italic_imgs[idx], cmap="gray", vmin=0, vmax=1)
            ax.set_title(f"'{italic_labels[idx]}'", fontsize=6, pad=1)
        ax.set_axis_off()
    fig.suptitle(f"ITALIC characters (showing {n_show}/{len(italic_imgs)})",
                 fontsize=12, fontweight="bold", color="#e74c3c")
    fig.tight_layout()
    plt.show()
else:
    print("No italic characters on this page.")

In [ ]:
# ── Grid of round characters ──────────────────────────────────────────
n_show = min(len(round_imgs), MAX_SHOW)
n_rows = max(1, int(np.ceil(n_show / n_cols)))

if n_show > 0:
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 0.8, n_rows * 1.0))
    axes = np.atleast_2d(axes)
    for idx in range(n_rows * n_cols):
        r, c = divmod(idx, n_cols)
        ax = axes[r, c]
        if idx < n_show:
            ax.imshow(round_imgs[idx], cmap="gray", vmin=0, vmax=1)
            ax.set_title(f"'{round_labels[idx]}'", fontsize=6, pad=1)
        ax.set_axis_off()
    fig.suptitle(f"ROUND characters (showing {n_show}/{len(round_imgs)})",
                 fontsize=12, fontweight="bold", color="#3498db")
    fig.tight_layout()
    plt.show()
else:
    print("No round characters on this page.")

---
## 4. Single word inspection

Select a word to see its characters and their italic labels.

In [ ]:
# ── Load per-word char bboxes ─────────────────────────────────────────
def load_word_chars(data, n_words):
    char_tblrs_list = []
    for i in range(n_words):
        key = f"char_tblrs_{i}"
        ct = data[key] if key in data else np.zeros((0, 4), dtype=np.int32)
        char_tblrs_list.append(ct)
    return char_tblrs_list

char_tblrs_list = load_word_chars(page_data, n_words)

In [ ]:
# ── Select word ───────────────────────────────────────────────────────
WORD_INDEX = 50  # change this

In [ ]:
wi = WORD_INDEX
word = words[wi]
wt, wb, wl, wr = word["tblr"]
text = word.get("text", "")
tblrs = char_tblrs_list[wi]
stroke = word_stroke_orientations[wi]

# Get chars for this word
word_mask = char_word_idx == wi
word_char_imgs = char_imgs[word_mask]
word_char_labels = char_labels_ocr[word_mask]
word_char_italic = char_italic_page[word_mask]
n_chars = len(word_char_imgs)

# Crop region
pad = max(1, (wb - wt) // 5)
crop_t = max(0, wt - pad)
crop_b = min(img_rgb.shape[0], wb + pad)
crop_l = max(0, wl - pad)
crop_r = min(img_rgb.shape[1], wr + pad)
word_crop = img_rgb[crop_t:crop_b, crop_l:crop_r]

# Plot
n_cols = max(n_chars, 1)
fig = plt.figure(figsize=(max(14, n_cols * 1.5), 5))

# Top: word crop with char bboxes colored by italic
ax_crop = fig.add_axes([0.02, 0.45, 0.96, 0.50])
ax_crop.imshow(word_crop)
for ci, tblr in enumerate(tblrs):
    t, b, l, r = tblr.astype(int)
    is_it = word_char_italic[ci] if ci < len(word_char_italic) else 0
    color = COLOR_ITALIC if is_it else COLOR_ROUND
    rect = Rectangle((l - crop_l, t - crop_t), r - l, b - t,
                      linewidth=2, edgecolor=color, facecolor="none")
    ax_crop.add_patch(rect)
ax_crop.set_axis_off()

word_class = "ITALIC" if word_is_italic[wi] else "ROUND"
ax_crop.set_title(
    f"Word {wi}: \"{text}\" — {word_class}  |  stroke={stroke:.2f}°  |  {n_chars} chars",
    fontsize=11, fontweight="bold",
    color=COLOR_ITALIC if word_is_italic[wi] else COLOR_ROUND,
)

# Bottom: embedded char images with italic labels
if n_chars > 0:
    for ci in range(n_chars):
        ax_ch = fig.add_axes([
            0.02 + ci * (0.96 / n_cols),
            0.02,
            0.96 / n_cols - 0.005,
            0.35,
        ])
        ax_ch.imshow(word_char_imgs[ci], cmap="gray", vmin=0, vmax=1)
        lbl = word_char_labels[ci] if ci < len(word_char_labels) else "?"
        is_it = word_char_italic[ci] if ci < len(word_char_italic) else 0
        color = COLOR_ITALIC if is_it else COLOR_ROUND
        ax_ch.set_title(f"'{lbl}'", fontsize=9, color=color)
        ax_ch.set_axis_off()

plt.show()

---
## 5. Browse pages

In [ ]:
print(f"Pages in document '{DOCUMENT}':")
offset = 0
for pname, pcount in zip(page_names, page_char_counts):
    if pcount == 0:
        print(f"  {pname:20s}  (no chars)")
        continue
    page_it = char_italic_all[offset:offset + pcount]
    n_it = page_it.sum()
    pct = 100 * n_it / pcount
    marker = "*" if pct > 20 else " "
    print(f"{marker} {pname:20s}  {pcount:4d} chars, {n_it:4d} italic ({pct:5.1f}%)")
    offset += pcount
print("\n* = pages with >20% italic")